Date table creation


In [0]:
from pyspark.sql.functions import min, max
df=spark.sql("select * from fmcg.gold.fact_orders")
display(df)


In [0]:
date_range=df.select(min('date'),max('date')).collect()[0]


In [0]:
display(date_range)

In [0]:
start_date=date_range[0]
end_date=date_range[1]
display (start_date,end_date)

In [0]:
from pyspark.sql.functions import sequence,explode,lit,to_date
date_df=spark.sql(f"""
                  select explode(sequence(to_date('{start_date}'),to_date('{end_date}'),interval 1 month)) as date
                  """)

display(date_df)

In [0]:

from pyspark.sql.functions import year, month, dayofmonth, quarter, weekofyear, date_format

date_df = date_df \
    .withColumn("date_key", date_format("date","yyyyMM").cast("int")) \
    .withColumn("year", year("date")) \
    .withColumn("month", month("date")) \
    .withColumn("day", dayofmonth("date")) \
    .withColumn("quarter", quarter("date")) \
    .withColumn("week", weekofyear("date")) \
    .withColumn("month_name", date_format("date", "MMMM")) \
    .withColumn("day_name", date_format("date", "EEEE"))
display(date_df)

In [0]:
display(date_df)

In [0]:
date_df.write.mode('overwrite').option('overwriteSchema','true').format('delta').saveAsTable("fmcg.gold.dim_date")

In [0]:
from pyspark.sql.functions import col

df.select('date').filter(col('date').isNull()).show()